# Colab T4 Inference Bottleneck Profiler (AD + LLM + E2E)

이 노트북은 **하나의 파일에서** 아래를 모두 측정합니다.
- AD 추론 병목 (`scripts/run_ad_inference.py`)
- LLM 추론 병목 (`src/mllm/*`)
- E2E 병목 (`scripts/run_experiment.py`)
- 모델 용량/메모리/latency 분해 및 결과 저장

출력물은 `output/profiling/colab_t4_bottleneck/`에 저장됩니다.


## 0) 사용 방법

1. Colab에서 드라이브 마운트 후 실행합니다.
2. 아래 `PROJECT_ROOT`, `DATA_ROOT`, `AD_CHECKPOINT_DIR` 경로를 확인합니다.
3. `RUN_*` 토글로 측정 섹션을 켜고/끄며 실행합니다.
4. 마지막 셀의 출력물(`*.csv`, `*.json`, `*.log`)을 저에게 전달하면 다음 최적화를 진행합니다.


In [ ]:
from pathlib import Path
import os

# ====== 필수 경로 ======
PROJECT_ROOT = Path("/content/drive/MyDrive/likelion/final_project/multimodal-anomaly-report-generation")
DATA_ROOT = Path("/content/drive/MyDrive/likelion/final_project/MMAD/dataset/MMAD")
MMAD_JSON = DATA_ROOT / "mmad.json"
AD_CHECKPOINT_DIR = Path("/content/drive/MyDrive/likelion/final_project/MMAD/checkpoints/patchcore_384")

# ====== 출력 경로 ======
OUTPUT_ROOT = PROJECT_ROOT / "output" / "profiling" / "colab_t4_bottleneck"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# ====== 샘플링 ======
SAMPLE_PER_FOLDER = 1
MAX_IMAGES = 80
SAMPLE_SEED = 42

# ====== 섹션 토글 ======
RUN_AD_SWEEP = True
RUN_LLM_SWEEP = True
RUN_E2E_SWEEP = True

# ====== LLM 기본 ======
DEFAULT_LLM_MODEL = "internvl3.5-2b"  # 예: internvl3.5-2b, qwen-2b, qwen
DEFAULT_FEW_SHOT = 1
DEFAULT_SIMILAR_TEMPLATE = False

os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT={PROJECT_ROOT}")
print(f"OUTPUT_ROOT={OUTPUT_ROOT}")


In [ ]:
# 선택: 최초 1회 의존성 설치가 필요하면 주석 해제
# !pip -q install -r requirements.txt

import gc
import io
import json
import os
import platform
import random
import re
import subprocess
import sys
import time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd

try:
    import torch
except Exception:
    torch = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None


In [ ]:
def _now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def _safe_div(a: float, b: float) -> float:
    return float(a) / float(b) if b else 0.0


def _pct(values: List[float], p: float) -> float:
    if not values:
        return 0.0
    return float(np.percentile(np.asarray(values, dtype=np.float64), p))


def summarize_latency_ms(values: List[float]) -> Dict[str, float]:
    return {
        "count": int(len(values)),
        "mean_ms": float(np.mean(values)) if values else 0.0,
        "p50_ms": _pct(values, 50),
        "p95_ms": _pct(values, 95),
        "p99_ms": _pct(values, 99),
        "min_ms": float(np.min(values)) if values else 0.0,
        "max_ms": float(np.max(values)) if values else 0.0,
    }


def maybe_cuda_sync() -> None:
    if torch is not None and torch.cuda.is_available():
        torch.cuda.synchronize()


def gpu_mem_snapshot() -> Dict[str, float]:
    if torch is None or not torch.cuda.is_available():
        return {
            "gpu_alloc_mb": 0.0,
            "gpu_reserved_mb": 0.0,
            "gpu_max_alloc_mb": 0.0,
        }
    return {
        "gpu_alloc_mb": float(torch.cuda.memory_allocated() / (1024 ** 2)),
        "gpu_reserved_mb": float(torch.cuda.memory_reserved() / (1024 ** 2)),
        "gpu_max_alloc_mb": float(torch.cuda.max_memory_allocated() / (1024 ** 2)),
    }


def reset_gpu_peak() -> None:
    if torch is not None and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def run_subprocess(cmd: List[str], cwd: Path) -> Dict[str, Any]:
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, capture_output=True)
    elapsed = time.perf_counter() - t0
    return {
        "returncode": int(proc.returncode),
        "stdout": proc.stdout,
        "stderr": proc.stderr,
        "elapsed_sec": float(elapsed),
        "cmd": cmd,
    }


In [ ]:
# 런타임 메타데이터 수집
runtime_meta = {
    "timestamp_utc": _now_iso(),
    "python": sys.version,
    "platform": platform.platform(),
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "mmad_json": str(MMAD_JSON),
    "ad_checkpoint_dir": str(AD_CHECKPOINT_DIR),
}

if torch is not None:
    runtime_meta.update({
        "torch_version": torch.__version__,
        "cuda_available": bool(torch.cuda.is_available()),
        "cuda_device_count": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    })

# nvidia-smi 캡처
try:
    smi = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
    runtime_meta["nvidia_smi"] = smi.stdout
except Exception as e:
    runtime_meta["nvidia_smi"] = f"unavailable: {e}"

# git 정보
try:
    branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"], cwd=str(PROJECT_ROOT), text=True).strip()
    commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=str(PROJECT_ROOT), text=True).strip()
    runtime_meta["git_branch"] = branch
    runtime_meta["git_commit"] = commit
except Exception:
    pass

meta_path = OUTPUT_ROOT / "runtime_meta.json"
meta_path.write_text(json.dumps(runtime_meta, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {meta_path}")
print(json.dumps({k: runtime_meta[k] for k in runtime_meta if k != 'nvidia_smi'}, indent=2, ensure_ascii=False))


In [ ]:
# MMAD 샘플 로드 및 고정 샘플셋 생성
with open(MMAD_JSON, "r", encoding="utf-8") as f:
    mmad_data = json.load(f)

all_image_paths = list(mmad_data.keys())


def stratified_sample(image_paths: List[str], n_per_folder: int, seed: int = 42) -> List[str]:
    rng = random.Random(seed)
    folders = defaultdict(list)
    for path in image_paths:
        parts = path.split("/")
        if len(parts) >= 4:
            key = f"{parts[0]}/{parts[1]}/{parts[2]}"  # dataset/category/split
        elif len(parts) >= 2:
            key = f"{parts[0]}/{parts[1]}"
        else:
            key = "unknown"
        folders[key].append(path)

    sampled = []
    for key in sorted(folders.keys()):
        imgs = folders[key]
        sampled.extend(rng.sample(imgs, min(n_per_folder, len(imgs))))
    return sampled

sampled_paths = stratified_sample(all_image_paths, SAMPLE_PER_FOLDER, SAMPLE_SEED)
if MAX_IMAGES:
    sampled_paths = sampled_paths[:MAX_IMAGES]

sampled_mmad = {k: mmad_data[k] for k in sampled_paths}
sampled_mmad_json = OUTPUT_ROOT / "sampled_mmad.json"
sampled_mmad_json.write_text(json.dumps(sampled_mmad, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Total images in mmad: {len(all_image_paths)}")
print(f"Sampled images: {len(sampled_paths)}")
print(f"Saved: {sampled_mmad_json}")


In [ ]:
# AD 체크포인트 용량 요약
ckpt_rows = []
for ckpt in AD_CHECKPOINT_DIR.rglob("model.ckpt"):
    rel = ckpt.relative_to(AD_CHECKPOINT_DIR)
    parts = rel.parts
    dataset = parts[1] if len(parts) > 1 else "unknown"
    category = parts[2] if len(parts) > 2 else "unknown"
    version = parts[3] if len(parts) > 3 else "unknown"
    size_mb = ckpt.stat().st_size / (1024 ** 2)
    ckpt_rows.append({
        "ckpt_path": str(ckpt),
        "dataset": dataset,
        "category": category,
        "version": version,
        "size_mb": size_mb,
    })

ckpt_df = pd.DataFrame(ckpt_rows).sort_values(["dataset", "category", "version"]) if ckpt_rows else pd.DataFrame()
ckpt_csv = OUTPUT_ROOT / "ad_checkpoint_sizes.csv"
ckpt_json = OUTPUT_ROOT / "ad_checkpoint_sizes.json"

if not ckpt_df.empty:
    ckpt_df.to_csv(ckpt_csv, index=False)
    ckpt_df.to_json(ckpt_json, orient="records", force_ascii=False, indent=2)
    print(f"checkpoint files: {len(ckpt_df)}")
    print(f"total size: {ckpt_df['size_mb'].sum():.2f} MB")
    display(ckpt_df.head(20))
    print(f"Saved: {ckpt_csv}")
else:
    print(f"No model.ckpt found under: {AD_CHECKPOINT_DIR}")


## 1) AD 병목 측정

`run_ad_inference.py`를 여러 설정으로 실행하고 아래를 수집합니다.
- wall time
- 처리 이미지 수/오류 수
- stage별 ms/img(read, infer, build, save)
- 평균 ms/img(스크립트 출력)


In [ ]:
AD_CASES = [
    {"name": "baseline", "batch_size": 8, "io_workers": 8, "decode_reduced": 1, "amp": True,  "postprocess_map": "input"},
    {"name": "small_batch", "batch_size": 4, "io_workers": 8, "decode_reduced": 1, "amp": True,  "postprocess_map": "input"},
    {"name": "large_batch", "batch_size": 16, "io_workers": 8, "decode_reduced": 1, "amp": True,  "postprocess_map": "input"},
    {"name": "less_io", "batch_size": 8, "io_workers": 2, "decode_reduced": 1, "amp": True,  "postprocess_map": "input"},
    {"name": "decode_x2", "batch_size": 8, "io_workers": 8, "decode_reduced": 2, "amp": True,  "postprocess_map": "input"},
    {"name": "no_amp", "batch_size": 8, "io_workers": 8, "decode_reduced": 1, "amp": False, "postprocess_map": "input"},
    {"name": "post_original", "batch_size": 8, "io_workers": 8, "decode_reduced": 1, "amp": True,  "postprocess_map": "original"},
]

_profile_re = re.compile(
    r"\[profile(?:-tail)?\]\s+(?P<imgs>\d+)\s+imgs\s+\|\s+read\s+(?P<read>[0-9.]+)\s+ms/img\s+\|\s+infer\s+(?P<infer>[0-9.]+)\s+ms/img\s+\|\s+build\s+(?P<build>[0-9.]+)\s+ms/img\s+\|\s+save\s+(?P<save>[0-9.]+)\s+ms/img"
)


def parse_ad_profile(stdout: str) -> Dict[str, float]:
    chunks = []
    for m in _profile_re.finditer(stdout):
        chunks.append({
            "imgs": int(m.group("imgs")),
            "read_ms": float(m.group("read")),
            "infer_ms": float(m.group("infer")),
            "build_ms": float(m.group("build")),
            "save_ms": float(m.group("save")),
        })
    if not chunks:
        return {"read_ms": 0.0, "infer_ms": 0.0, "build_ms": 0.0, "save_ms": 0.0}

    w = float(sum(c["imgs"] for c in chunks))
    return {
        "read_ms": sum(c["read_ms"] * c["imgs"] for c in chunks) / w,
        "infer_ms": sum(c["infer_ms"] * c["imgs"] for c in chunks) / w,
        "build_ms": sum(c["build_ms"] * c["imgs"] for c in chunks) / w,
        "save_ms": sum(c["save_ms"] * c["imgs"] for c in chunks) / w,
    }


def run_ad_case(case: Dict[str, Any]) -> Dict[str, Any]:
    name = case["name"]
    ad_out = OUTPUT_ROOT / f"ad_predictions_{name}.json"
    ad_log = OUTPUT_ROOT / f"ad_profile_{name}.log"

    cmd = [
        sys.executable,
        "scripts/run_ad_inference.py",
        "--backend", "ckpt",
        "--checkpoint-dir", str(AD_CHECKPOINT_DIR),
        "--data-root", str(DATA_ROOT),
        "--mmad-json", str(sampled_mmad_json),
        "--output", str(ad_out),
        "--output-format", "report",
        "--config", "configs/anomaly.yaml",
        "--device", "cuda",
        "--batch-size", str(case["batch_size"]),
        "--io-workers", str(case["io_workers"]),
        "--decode-reduced", str(case["decode_reduced"]),
        "--postprocess-map", str(case["postprocess_map"]),
        "--profile-interval", "32",
        "--threshold", "0.5",
    ]
    cmd.append("--amp" if case["amp"] else "--no-amp")

    result = run_subprocess(cmd, PROJECT_ROOT)
    ad_log.write_text(result["stdout"] + "[stderr]" + result["stderr"], encoding="utf-8")

    stage = parse_ad_profile(result["stdout"])
    processed_m = re.search(r"Processed \(this run\):\s*(\d+)", result["stdout"])
    err_m = re.search(r"Errors:\s*(\d+)", result["stdout"])
    avg_m = re.search(r"Average:\s*([0-9.]+)ms/img", result["stdout"])

    processed = int(processed_m.group(1)) if processed_m else 0
    errors = int(err_m.group(1)) if err_m else 0
    script_avg_ms = float(avg_m.group(1)) if avg_m else 0.0

    row = {
        "section": "ad",
        "case": name,
        "returncode": result["returncode"],
        "elapsed_sec": result["elapsed_sec"],
        "processed": processed,
        "errors": errors,
        "script_avg_ms": script_avg_ms,
        "throughput_img_s": _safe_div(processed, result["elapsed_sec"]),
        "stage_read_ms": stage["read_ms"],
        "stage_infer_ms": stage["infer_ms"],
        "stage_build_ms": stage["build_ms"],
        "stage_save_ms": stage["save_ms"],
        "output_json": str(ad_out),
        "log_path": str(ad_log),
        **case,
    }
    return row


In [ ]:
ad_results = []
if RUN_AD_SWEEP:
    for case in AD_CASES:
        print(f"[AD] running: {case['name']}")
        row = run_ad_case(case)
        ad_results.append(row)
        print({k: row[k] for k in ["case", "returncode", "elapsed_sec", "processed", "script_avg_ms", "throughput_img_s"]})

ad_df = pd.DataFrame(ad_results)
ad_csv = OUTPUT_ROOT / "ad_profile_summary.csv"
ad_json = OUTPUT_ROOT / "ad_profile_summary.json"
if not ad_df.empty:
    ad_df.to_csv(ad_csv, index=False)
    ad_df.to_json(ad_json, orient="records", force_ascii=False, indent=2)
    display(ad_df.sort_values("script_avg_ms"))
    print(f"Saved: {ad_csv}")
else:
    print("AD sweep skipped or no results.")


## 2) LLM 병목 측정

이 섹션은 로컬 LLM 기준으로 다음을 수집합니다.
- 모델 로드 시간 (cold)
- 이미지당 추론 지연 (official API: `generate_answers` / `generate_answers_batch`)
- 질문당 지연
- 오류율
- GPU 메모리 스냅샷


In [ ]:
from src.mllm.factory import get_llm_client


def _select_few_shot(meta: Dict[str, Any], few_shot: int, similar_template: bool) -> List[str]:
    key = "similar_templates" if similar_template else "random_templates"
    return list(meta.get(key, []))[:few_shot]


def profile_llm_case(case: Dict[str, Any], image_paths: List[str]) -> Dict[str, Any]:
    reset_gpu_peak()
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

    model_name = case["model"]
    client = get_llm_client(model_name, max_image_size=tuple(case["max_image_size"]))
    if hasattr(client, "max_new_tokens"):
        client.max_new_tokens = int(case["max_new_tokens"])

    # cold load
    t_load0 = time.perf_counter()
    if hasattr(client, "_load_model"):
        try:
            client._load_model()
        except Exception:
            pass
    maybe_cuda_sync()
    load_sec = time.perf_counter() - t_load0

    mem_after_load = gpu_mem_snapshot()

    per_image_ms = []
    per_question_ms = []
    question_counts = []
    failures = 0

    for image_rel in image_paths:
        meta = mmad_data[image_rel]
        query_image_path = str(DATA_ROOT / image_rel)
        if not Path(query_image_path).exists():
            failures += 1
            continue

        few_shot_rel = _select_few_shot(meta, case["few_shot"], case["similar_template"])
        few_shot_paths = [str(DATA_ROOT / p) for p in few_shot_rel if Path(DATA_ROOT / p).exists()]

        # 질문 수 계산
        q_list, gt_answers, q_types = client.parse_conversation(meta)
        qn = len(gt_answers)
        if qn == 0:
            failures += 1
            continue

        t0 = time.perf_counter()
        try:
            if case["batch_mode"]:
                _, answers, predicted, _ = client.generate_answers_batch(
                    query_image_path, meta, few_shot_paths, ad_info=None, instruction=None
                )
            else:
                _, answers, predicted, _ = client.generate_answers(
                    query_image_path, meta, few_shot_paths, ad_info=None, instruction=None
                )
        except Exception:
            predicted = None
            answers = gt_answers
        maybe_cuda_sync()
        dt_ms = (time.perf_counter() - t0) * 1000.0

        ok = predicted is not None and len(predicted) == len(answers)
        if not ok:
            failures += 1
            continue

        per_image_ms.append(dt_ms)
        question_counts.append(qn)
        per_question_ms.append(_safe_div(dt_ms, qn))

    summary = summarize_latency_ms(per_image_ms)
    q_summary = summarize_latency_ms(per_question_ms)

    row = {
        "section": "llm",
        "case": case["name"],
        "model": model_name,
        "batch_mode": bool(case["batch_mode"]),
        "max_image_size": str(tuple(case["max_image_size"])),
        "max_new_tokens": int(case["max_new_tokens"]),
        "few_shot": int(case["few_shot"]),
        "similar_template": bool(case["similar_template"]),
        "load_sec": load_sec,
        "num_images": int(len(per_image_ms)),
        "failures": int(failures),
        "failure_rate": _safe_div(failures, len(image_paths)),
        "img_mean_ms": summary["mean_ms"],
        "img_p50_ms": summary["p50_ms"],
        "img_p95_ms": summary["p95_ms"],
        "img_p99_ms": summary["p99_ms"],
        "q_mean_ms": q_summary["mean_ms"],
        "q_p95_ms": q_summary["p95_ms"],
        "avg_questions_per_image": float(np.mean(question_counts)) if question_counts else 0.0,
        **mem_after_load,
    }

    # 메모리 정리
    del client
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row


In [ ]:
LLM_CASES = [
    {
        "name": "llm_baseline",
        "model": DEFAULT_LLM_MODEL,
        "batch_mode": True,
        "max_image_size": [384, 384],
        "max_new_tokens": 128,
        "few_shot": DEFAULT_FEW_SHOT,
        "similar_template": DEFAULT_SIMILAR_TEMPLATE,
    },
    {
        "name": "llm_small_image",
        "model": DEFAULT_LLM_MODEL,
        "batch_mode": True,
        "max_image_size": [256, 256],
        "max_new_tokens": 128,
        "few_shot": DEFAULT_FEW_SHOT,
        "similar_template": DEFAULT_SIMILAR_TEMPLATE,
    },
    {
        "name": "llm_short_gen",
        "model": DEFAULT_LLM_MODEL,
        "batch_mode": True,
        "max_image_size": [384, 384],
        "max_new_tokens": 64,
        "few_shot": DEFAULT_FEW_SHOT,
        "similar_template": DEFAULT_SIMILAR_TEMPLATE,
    },
    {
        "name": "llm_incremental",
        "model": DEFAULT_LLM_MODEL,
        "batch_mode": False,
        "max_image_size": [384, 384],
        "max_new_tokens": 128,
        "few_shot": DEFAULT_FEW_SHOT,
        "similar_template": DEFAULT_SIMILAR_TEMPLATE,
    },
]

llm_results = []
if RUN_LLM_SWEEP:
    for case in LLM_CASES:
        print(f"[LLM] running: {case['name']} ({case['model']})")
        row = profile_llm_case(case, sampled_paths)
        llm_results.append(row)
        print({k: row[k] for k in ["case", "num_images", "img_mean_ms", "img_p95_ms", "failure_rate"]})

llm_df = pd.DataFrame(llm_results)
llm_csv = OUTPUT_ROOT / "llm_profile_summary.csv"
llm_json = OUTPUT_ROOT / "llm_profile_summary.json"
if not llm_df.empty:
    llm_df.to_csv(llm_csv, index=False)
    llm_df.to_json(llm_json, orient="records", force_ascii=False, indent=2)
    display(llm_df.sort_values("img_mean_ms"))
    print(f"Saved: {llm_csv}")
else:
    print("LLM sweep skipped or no results.")


## 3) E2E 병목 측정

`run_experiment.py`를 호출하여 E2E wall time / accuracy / 처리량을 비교합니다.
- `llm_only`
- `ad_plus_llm` (`--ad-output`로 AD 재사용)


In [ ]:
# AD baseline 산출물 경로 탐색 (없으면 E2E AD+LLM 케이스 skip)
def find_ad_baseline_output(ad_df: pd.DataFrame) -> Optional[str]:
    if ad_df is None or ad_df.empty:
        return None
    hit = ad_df[ad_df["case"] == "baseline"]
    if hit.empty:
        return None
    p = hit.iloc[0].get("output_json")
    if p and Path(p).exists():
        return str(p)
    return None

ad_baseline_output = find_ad_baseline_output(ad_df if 'ad_df' in globals() else pd.DataFrame())
print("ad_baseline_output:", ad_baseline_output)


In [ ]:
E2E_CASES = [
    {
        "name": "e2e_llm_only",
        "llm": DEFAULT_LLM_MODEL,
        "ad_model": "null",
        "batch_mode": True,
        "few_shot": DEFAULT_FEW_SHOT,
        "ad_output": None,
    },
    {
        "name": "e2e_ad_plus_llm",
        "llm": DEFAULT_LLM_MODEL,
        "ad_model": "patchcore",
        "batch_mode": True,
        "few_shot": DEFAULT_FEW_SHOT,
        "ad_output": ad_baseline_output,
    },
]


def run_e2e_case(case: Dict[str, Any]) -> Dict[str, Any]:
    out_dir = OUTPUT_ROOT / "e2e_runs" / case["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / "run_experiment.log"

    cmd = [
        sys.executable,
        "scripts/run_experiment.py",
        "--config", "configs/experiment.yaml",
        "--llm", case["llm"],
        "--ad-model", case["ad_model"],
        "--data-root", str(DATA_ROOT),
        "--mmad-json", str(sampled_mmad_json),
        "--output-dir", str(out_dir),
        "--few-shot", str(case["few_shot"]),
        "--batch-mode", "true" if case["batch_mode"] else "false",
        "--max-images", str(len(sampled_paths)),
    ]

    if case.get("ad_output"):
        cmd.extend(["--ad-output", str(case["ad_output"])])

    result = run_subprocess(cmd, PROJECT_ROOT)
    log_path.write_text(result["stdout"] + "

[stderr]
" + result["stderr"], encoding="utf-8")

    # meta 파일 읽기
    meta_files = sorted(out_dir.glob("*.meta.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    meta = {}
    if meta_files:
        with open(meta_files[0], "r", encoding="utf-8") as f:
            meta = json.load(f)

    row = {
        "section": "e2e",
        "case": case["name"],
        "returncode": result["returncode"],
        "elapsed_wall_sec": result["elapsed_sec"],
        "meta_elapsed_sec": float(meta.get("elapsed_seconds", 0.0)) if meta else 0.0,
        "processed": int(meta.get("processed", 0)) if meta else 0,
        "errors": int(meta.get("errors", 0)) if meta else 0,
        "total_questions": int(meta.get("total_questions", 0)) if meta else 0,
        "accuracy": float(meta.get("accuracy", 0.0)) if meta else 0.0,
        "throughput_img_s": _safe_div(int(meta.get("processed", 0)) if meta else 0, result["elapsed_sec"]),
        "llm": case["llm"],
        "ad_model": case["ad_model"],
        "batch_mode": bool(case["batch_mode"]),
        "few_shot": int(case["few_shot"]),
        "ad_output": case.get("ad_output"),
        "log_path": str(log_path),
        "output_dir": str(out_dir),
    }
    return row


In [ ]:
e2e_results = []
if RUN_E2E_SWEEP:
    for case in E2E_CASES:
        if case["ad_model"] != "null" and not case.get("ad_output"):
            print(f"[E2E] skip {case['name']} (ad_output not found)")
            continue
        print(f"[E2E] running: {case['name']}")
        row = run_e2e_case(case)
        e2e_results.append(row)
        print({k: row[k] for k in ["case", "returncode", "elapsed_wall_sec", "processed", "accuracy"]})

e2e_df = pd.DataFrame(e2e_results)
e2e_csv = OUTPUT_ROOT / "e2e_profile_summary.csv"
e2e_json = OUTPUT_ROOT / "e2e_profile_summary.json"
if not e2e_df.empty:
    e2e_df.to_csv(e2e_csv, index=False)
    e2e_df.to_json(e2e_json, orient="records", force_ascii=False, indent=2)
    display(e2e_df.sort_values("elapsed_wall_sec"))
    print(f"Saved: {e2e_csv}")
else:
    print("E2E sweep skipped or no results.")


## 4) 병목 요약

- AD stage 비중 계산
- LLM case별 latency 비교
- E2E 비교


In [ ]:
summary_rows = []

if 'ad_df' in globals() and not ad_df.empty:
    ad_tmp = ad_df.copy()
    ad_tmp["ad_stage_total_ms"] = ad_tmp[["stage_read_ms", "stage_infer_ms", "stage_build_ms", "stage_save_ms"]].sum(axis=1)
    ad_tmp["infer_share"] = ad_tmp["stage_infer_ms"] / ad_tmp["ad_stage_total_ms"].replace(0, np.nan)
    ad_tmp["read_share"] = ad_tmp["stage_read_ms"] / ad_tmp["ad_stage_total_ms"].replace(0, np.nan)
    ad_tmp["build_share"] = ad_tmp["stage_build_ms"] / ad_tmp["ad_stage_total_ms"].replace(0, np.nan)
    ad_tmp["save_share"] = ad_tmp["stage_save_ms"] / ad_tmp["ad_stage_total_ms"].replace(0, np.nan)
    display(ad_tmp[["case", "script_avg_ms", "stage_read_ms", "stage_infer_ms", "stage_build_ms", "stage_save_ms", "infer_share", "read_share", "build_share", "save_share"]].sort_values("script_avg_ms"))

if 'llm_df' in globals() and not llm_df.empty:
    display(llm_df[["case", "model", "batch_mode", "img_mean_ms", "img_p95_ms", "q_mean_ms", "failure_rate", "load_sec"]].sort_values("img_mean_ms"))

if 'e2e_df' in globals() and not e2e_df.empty:
    display(e2e_df[["case", "llm", "ad_model", "elapsed_wall_sec", "processed", "accuracy", "throughput_img_s"]].sort_values("elapsed_wall_sec"))

if plt is not None and 'ad_df' in globals() and not ad_df.empty:
    best = ad_df.sort_values("script_avg_ms").iloc[0]
    labels = ["read", "infer", "build", "save"]
    values = [best["stage_read_ms"], best["stage_infer_ms"], best["stage_build_ms"], best["stage_save_ms"]]
    plt.figure(figsize=(6, 4))
    plt.bar(labels, values)
    plt.title(f"AD Stage Breakdown (best={best['case']})")
    plt.ylabel("ms/img")
    plt.show()

summary = {
    "timestamp_utc": _now_iso(),
    "output_root": str(OUTPUT_ROOT),
    "ad_cases": len(ad_df) if 'ad_df' in globals() else 0,
    "llm_cases": len(llm_df) if 'llm_df' in globals() else 0,
    "e2e_cases": len(e2e_df) if 'e2e_df' in globals() else 0,
}
summary_path = OUTPUT_ROOT / "profiling_summary.json"
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved: {summary_path}")


In [ ]:
# 결과 파일 목록
for p in sorted(OUTPUT_ROOT.glob("**/*")):
    if p.is_file():
        print(p.relative_to(OUTPUT_ROOT))
